In [206]:
import sys
import os

sys.path.append(os.path.abspath("../")) 

from data import DataModule, Vocabulary
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch

In [223]:
class MTEngUa(DataModule):
    def __init__(self, batch_size, num_steps, num_train=512, num_val=128):
        self.batch_size = batch_size
        self.num_steps = num_steps
        self.num_train = num_train
        self.num_val = num_val

        self.arrays, self.src_vocab, self.trg_vocab = self._build_arrays(*self._download()) 

    def _download(self):
        data =  pd.read_json("../books/eng-ua.json", lines=True).values
        return (data[:, 0], data[:, 1])
    
    def _preprocess(self, src, trg):

        def preprocess_sent(sent):
            add_space = lambda char, prev_char: char in ',.!?' and prev_char != " "

            preprosseced_sent = [" " + sent[i] if i > 0 and add_space(sent[i], sent[i - 1]) 
                                 else sent[i]
                                 for i in range(len(sent))]

            return "".join(preprosseced_sent).lower()

        preprosseced_src = [preprocess_sent(sent) for sent in src]
        preprosseced_trg = [preprocess_sent(sent) for sent in trg]
        
        return preprosseced_src, preprosseced_trg
    
    def _tokenize(self, src, trg):
        tokenized_src = [sent.split(' ') + ['<eos>'] for sent in src]
        tokenized_trg = [sent.split(' ') + ['<eos>'] for sent in trg]

        return tokenized_src, tokenized_trg
    
    def _build_arrays(self, src, trg):

        src, trg = self._tokenize(*self._preprocess(src, trg))

        pad_or_trim = lambda tokens: tokens[:self.num_steps] if len(tokens) > self.num_steps  \
                                                else tokens + ['<pad>'] * (self.num_steps - len(tokens))

        src = [pad_or_trim(tokens) for tokens in src]
        trg = [['<bos>'] + pad_or_trim(tokens) for tokens in trg]

        src_vocab = Vocabulary(text=[token for tokens in src for token in tokens])
        trg_vocab = Vocabulary(text=[token for tokens in trg for token in tokens])

        src = torch.tensor([src_vocab[tokens] for tokens in src])
        trg = torch.tensor([trg_vocab[tokens] for tokens in trg])

        src_valid_len = (src != src_vocab["<pad>"]).type(torch.int32).sum(1)

        return (src, trg[:, :-1], src_valid_len, trg[:, 1:]), src_vocab, trg_vocab
    
    def get_dataloader(self, train):
        idx = slice(0, self.num_train) if train else slice(self.num_train, None)
        
        return self.get_tensorloader(self.arrays, train, idx)
    

In [224]:
data = MTEngUa(batch_size=32, num_steps=21)

In [225]:
src, tgt, src_valid_len, label = next(iter(data.train_dataloader()))
print('source:', src.type(torch.int32))
print('decoder input:', tgt.type(torch.int32))
print('source len excluding pad:', src_valid_len.type(torch.int32))
print('label:', label.type(torch.int32))

source: tensor([[  19,   29,   15,   43,   44,   19,   45,   46,   39,   47,   29,   48,
           49,   31,   10,   11,   12,   12,   12,   12,   12],
        [  19,  738,  240,   34,   19,   97,   34,   19,  235,  589,   11,   12,
           12,   12,   12,   12,   12,   12,   12,   12,   12],
        [  19,   61,   62,  721,  495,  722,  723,  114,   55,  724,   10,   11,
           12,   12,   12,   12,   12,   12,   12,   12,   12],
        [   0,  118,    5,  108,   34,   19,  113,    2,   90,   84,   98,   32,
           33,   66,  409,   10,   11,   12,   12,   12,   12],
        [  14,   65,   59,   19,   60,   62,  146,   10,   11,   12,   12,   12,
           12,   12,   12,   12,   12,   12,   12,   12,   12],
        [  19,   61,   62,   19,   63,   64,   32,   65,   66,   19,   67,   10,
           11,   12,   12,   12,   12,   12,   12,   12,   12],
        [ 218,   29,  219,   39,  220,   10,   11,   12,   12,   12,   12,   12,
           12,   12,   12,   12,   12,   

In [194]:
# tokens_per_sent_src = [len(sent) for sent in src]
# tokens_per_sent_trg = [len(sent) for sent in trg]

# sns.histplot(tokens_per_sent_src, kde=True, color="skyblue", label="Source (English)")
# sns.histplot(tokens_per_sent_trg, kde=True, color="orange", label="Target (Ukrainian)")

# plt.title("Distribution of Sentence Lengths: Source vs Target")
# plt.xlabel("Number of Tokens")
# plt.ylabel("Frequency")

# plt.legend()
# plt.show()

In [195]:
# max_src_len = int(np.percentile(tokens_per_sent_src, 95))
# max_trg_len = int(np.percentile(tokens_per_sent_trg, 95))

# print(f"95% of English sentences are <= {max_src_len} tokens")
# print(f"95% of Ukrainian sentences are <= {max_trg_len} tokens")